In [19]:
import numpy as np

from caldis.core.system import System
from caldis.fluids import CoolPropBackend

from caldis.components import FluidBoundary
from caldis.components import Compressor

#SizingHX = recuperator
# SimpleHeater = inter /after-cooler

from caldis.components import SizingHX, SimpleHeater

#FlashTank
from caldis.components import FlashTank , Mixer , Valve

#only for L3
from caldis.components import Splitter
from caldis.solvers import solve        

from CoolProp.CoolProp import PropsSI

In [20]:

gas = "Methane"
be_methane = CoolPropBackend(gas)
T0 = 273.15 + 25.0  # K

T_IC = T0 + 5.0 # K
T_aftercooler = T0 + 5.0 # K


P_high = 100e5

comp_is_e = 0.75 
HX_pinch = 5 # K
HP_recup_coeff = 1000 # W/m2K
LP_recup_coeff = 500 # W/m2K

mdot_liq = 1 # kg/s

In [ ]:
source    = FluidBoundary("source", T=T0, backend=be_methane)

Mixer_ = Mixer("Mixer", backend=be_methane)
# Compressors (4 compressors in series, with intercooling between each stage)

Comp1 = Compressor("Comp1", eta_is=comp_is_e, backend=be_methane)
Comp2 = Compressor("Comp2", eta_is=comp_is_e, backend=be_methane)
Comp3 = Compressor("Comp3", eta_is=comp_is_e, backend=be_methane)
Comp4 = Compressor("Comp4", eta_is=comp_is_e, backend=be_methane)

IC1 = SimpleHeater("IC1", backend=be_methane)
IC2 = SimpleHeater("IC2",backend=be_methane)
IC3 = SimpleHeater("IC3",backend=be_methane)

Aftercooler = SimpleHeater("Aftercooler", T_source=T_aftercooler, backend=be_methane)

Recuperator = SizingHX("Recuperator", h_hot = HP_recup_coeff, h_cold = LP_recup_coeff, backend_hot=be_methane, backend_cold=be_methane, area_ratio_cold_to_hot=1.0)

#Valve     def __init__(self, name, Kv=None, backend=None, flow_law="quadratic", eps=1e-9):
valve = Valve("valve", backend=be_methane)

Flash = FlashTank("Flash", backend=be_methane)

liquid_product = FluidBoundary("liquid_product", backend=be_methane)


In [ ]:
sys = System("cycle")

sys.add( source, Comp1, IC1, Comp2, IC2, Comp3, IC3, Comp4, Aftercooler, Recuperator, valve, Flash)

sys.connect(source.port, Mixer_.in1)
#afficher les attributs de Comp1
#print(Comp1.attributes#)

sys.connect(Mixer_.out, Comp1.inl)
sys.connect(Comp1.out, IC1.inl)
sys.connect(IC1.out, Comp2.inl)
sys.connect(Comp2.out, IC2.inl)
sys.connect(IC2.out, Comp3.inl)
sys.connect(Comp3.out, IC3.inl)
sys.connect(IC3.out, Comp4.inl)


sys.connect(Comp4.out, Aftercooler.inl)
sys.connect(Aftercooler.out, Recuperator.hot_in)



sys.connect(Recuperator.hot_out, valve.inl)
sys.connect(valve.out, Flash.inl)

sys.connect(Recuperator.cold_in, Flash.vap)
sys.connect(Recuperator.cold_out, Mixer_.in2)

sys.connect(Flash.liq, liquid_product.port)

# add constraints on the compressors, r=3.16 for all stages
sys.add_constraint(lambda: Comp1.out.p - 3.16 * Comp1.inl.p, "fn=comp1_pressure_ratio")
sys.add_constraint(lambda: Comp2.out.p - 3.16 * Comp2.inl.p, "fn=comp2_pressure_ratio")
sys.add_constraint(lambda: Comp3.out.p - 3.16 * Comp3.inl.p, "fn=comp3_pressure_ratio")
sys.add_constraint(lambda: Comp4.out.p - 3.16 * Comp4.inl.p, "fn=comp4_pressure_ratio")

sys.add_constraint(lambda: Aftercooler.out.T - T_aftercooler, "fn=aftercooler_temperature")
sys.add_constraint(lambda: liquid_product.port.mdot - mdot_liq, "fn=liquid_product_mass_flow")

#sys.add_constraint(lambda : 
sol = solve(sys)



✗ DOF: [DOF sous-spécifié] 85 inconnues, 83 équations : il manque 2 condition(s) aux limites (variables non déterminées).
  -> the model is not square. Add/remove a constraint or boundary spec (MODELLING issue, not a start issue).


In [40]:
sys